# Entrenamiento de Red Neuronal - PyTorch

Este notebook desarrolla el proceso para entrenar una red neuronal, evaluar el sobreajuste usando TensorBoard, mitigarlo y comparar los resultados frente a los modelos clásicos (RF, KNN, Tree).

In [ ]:
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
#!pip install pandas scikit-learn tensorboard joblib statsmodels jupyter

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from statsmodels.stats.contingency_tables import mcnemar

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Preparación de los Datos
Vamos a cargar los datos y preprocesarlos utilizando las mismas funciones definidas en `Funciones.py` que se utilizaron para los modelos clásicos. Así aseguramos que la comparación sea justa.

In [3]:
import sys
sys.path.append('..')
import Funciones as f

# ── 1. Cargar y preprocesar ──────────────────────────────────────────────────
np.random.seed(42)

data = f.get_data('../games.csv')
data = f.filter(data, 'increment_code', 0.02)
data = f.filter(data, 'opening_eco', 0.02)
data = f.code(data)
data = f.balance(data)

# Recuperar las features seleccionadas por RFE del proyecto original
rf_data       = joblib.load("../chess_random_forest_model.joblib")
list_features = list(rf_data['feature_names'])

X = data[list_features]
y = data['winner']

# ── 2. Split PRIMERO, antes de cualquier fit ─────────────────────────────────
# Regla: nada aprende del dataset completo. El test set no existe hasta aquí.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── 3. Scaler: fit SOLO sobre train, transform sobre ambos ───────────────────
# Si se hace fit sobre X completo, el test set "filtra" información al scaler.
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)   # aprende media/std del train
X_test  = scaler.transform(X_test_raw)        # aplica sin re-aprender

# ── 4. Datasets de PyTorch ───────────────────────────────────────────────────
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test.values, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"Tamaño de entrenamiento : {len(train_dataset)}")
print(f"Tamaño de prueba        : {len(test_dataset)}")
print(f"Features utilizadas     : {len(list_features)}")

-----------------black -> 0------------------------
-----------------white -> 1------------------------
       white_rating  black_rating  opening_ply  victory_status_mate  \
3          0.356366      0.341805     0.166667                 True   
6          0.400435      0.325726     0.750000                False   
8          0.356366      0.309647     0.416667                False   
9          0.324810      0.214730     0.250000                 True   
13         0.324810      0.421162     0.083333                False   
...             ...           ...          ...                  ...   
20024      0.505441      0.566390     0.166667                 True   
20033      0.262786      0.237552     0.083333                False   
20049      0.295974      0.237033     0.333333                 True   
20055      0.236670      0.254668     0.166667                 True   
20057      0.245375      0.282158     0.166667                 True   

       victory_status_outoftime  victory_st

c:\Users\alvar\.gemini\antigravity\worktrees\PROYECTO_FIA\pytorch-training-analysis-report\red-neuronal\..\Funciones.py:64: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('winner', group_keys=False).apply(lambda x: x.sample(minimo_clase)).reset_index(drop=True)
c:\Users\alvar\.gemini\antigravity\worktrees\PROYECTO_FIA\pytorch-training-analysis-report\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.5.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#securit

## 2. Evaluación de Modelos Clásicos (Baseline)
Obtenemos las predicciones de los tres modelos para utilizarlas después en el Test de McNemar.

In [4]:
# ── Modelos clásicos reentrenados desde cero en el mismo pipeline ─────────────
#
# CORRECCIÓN: en el notebook anterior se cargaban modelos serializados del
# proyecto original (sklearn 1.5.1, entrenados sobre el dataset completo).
# Eso inflaba el accuracy de RF y Tree porque habían "visto" el test set
# durante su entrenamiento original.
#
# Aquí reentrenamos los tres modelos sobre X_train exclusivamente,
# usando los mismos hiperparámetros que el proyecto original.

# Random Forest — igual que en Funciones.py (sin hiperparámetros especiales)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_raw, y_train)

# Decision Tree — criterio Gini, igual que en Funciones.py
tree_model = DecisionTreeClassifier(criterion='gini', random_state=42)
tree_model.fit(X_train_raw, y_train)

# K-NN — k = sqrt(N) redondeado a impar, igual que en Funciones.py
k = int(np.sqrt(len(X_train)))
k = k if k % 2 != 0 else k + 1
knn_model = KNeighborsClassifier(n_neighbors=k)
knn_model.fit(X_train, y_train)   # K-NN usa datos escalados

# Predicciones sobre el test set limpio
preds_rf   = rf_model.predict(X_test_raw)   # árboles: sin scaler
preds_tree = tree_model.predict(X_test_raw)
preds_knn  = knn_model.predict(X_test)      # K-NN: con scaler

print(f"k utilizado para K-NN : {k}")
print(f"Accuracy RF           : {accuracy_score(y_test, preds_rf):.4f}")
print(f"Accuracy K-NN         : {accuracy_score(y_test, preds_knn):.4f}")
print(f"Accuracy Decision Tree: {accuracy_score(y_test, preds_tree):.4f}")

k utilizado para K-NN : 63
Accuracy RF           : 0.6519
Accuracy K-NN         : 0.6097
Accuracy Decision Tree: 0.6004


## 3. Red Neuronal (Propenso a Sobreajuste)
Definimos una red densa y compleja sin regularización para observar el overfitting en TensorBoard.

In [5]:
class ChessNN_Overfit(nn.Module):
    def __init__(self, input_size):
        super(ChessNN_Overfit, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )
        
    def forward(self, x):
        return self.net(x)

model_overfit = ChessNN_Overfit(X.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_overfit.parameters(), lr=1e-3)

writer = SummaryWriter('runs/chess_experiment_overfit')

epochs = 80
for epoch in range(epochs):
    model_overfit.train()
    total_loss, correct, total = 0, 0, 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model_overfit(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
    train_loss = total_loss / total
    train_acc = correct / total
    
    model_overfit.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model_overfit(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == batch_y).sum().item()
            
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    
    writer.add_scalars('Loss', {'Train': train_loss, 'Validation': val_loss}, epoch)
    writer.add_scalars('Accuracy', {'Train': train_acc, 'Validation': val_acc}, epoch)

writer.close()
print("Entrenamiento finalizado. Para ver el resultado:\nAbre una terminal en esta carpeta y ejecuta: tensorboard --logdir runs")

Entrenamiento finalizado. Para ver el resultado:
Abre una terminal en esta carpeta y ejecuta: tensorboard --logdir runs


## 4. Mitigación del Sobreajuste
Introducimos Dropout y Weight Decay (regularización L2).

In [6]:
class ChessNN_Reg(nn.Module):
    def __init__(self, input_size):
        super(ChessNN_Reg, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )
        
    def forward(self, x):
        return self.net(x)

model_reg = ChessNN_Reg(X.shape[1])
optimizer_reg = optim.Adam(model_reg.parameters(), lr=1e-3, weight_decay=1e-4) # L2 Penalty
writer_reg = SummaryWriter('runs/chess_experiment_regularized')

epochs = 80
for epoch in range(epochs):
    model_reg.train()
    total_loss, correct, total = 0, 0, 0
    for batch_X, batch_y in train_loader:
        optimizer_reg.zero_grad()
        outputs = model_reg(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer_reg.step()
        
        total_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
    train_loss = total_loss / total
    train_acc = correct / total
    
    model_reg.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model_reg(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == batch_y).sum().item()
            
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    
    writer_reg.add_scalars('Loss', {'Train': train_loss, 'Validation': val_loss}, epoch)
    writer_reg.add_scalars('Accuracy', {'Train': train_acc, 'Validation': val_acc}, epoch)

writer_reg.close()
print("Entrenamiento con regularización completado.")

Entrenamiento con regularización completado.


## 5. Comparación Estadística (Test de McNemar)
Aplicamos el test sobre las predicciones de los modelos para determinar si las diferencias son significativas.

In [7]:
model_reg.eval()
with torch.no_grad():
    outputs = model_reg(torch.tensor(X_test, dtype=torch.float32))
    _, preds_nn = torch.max(outputs.data, 1)
preds_nn = preds_nn.numpy()

def perform_mcnemar(model_name, preds_model):
    table = [[0, 0], [0, 0]]
    for nn_pred, m_pred, true_y in zip(preds_nn, preds_model, y_test.values):
        nn_correct = (nn_pred == true_y)
        m_correct = (m_pred == true_y)
        if nn_correct and m_correct:
            table[0][0] += 1
        elif nn_correct and not m_correct:
            table[0][1] += 1
        elif not nn_correct and m_correct:
            table[1][0] += 1
        else:
            table[1][1] += 1
            
    result = mcnemar(table, exact=True)
    print(f"--- McNemar Test: Red Neuronal vs {model_name} ---")
    print(f"P-value: {result.pvalue:.5f}")
    if result.pvalue < 0.05:
        print("Diferencia estadísticamente significativa.")
    else:
        print("No hay diferencia significativa.")
    print(f"Accuracy NN: {accuracy_score(y_test, preds_nn):.4f} | Accuracy {model_name}: {accuracy_score(y_test, preds_model):.4f}\n")

perform_mcnemar("Random Forest", preds_rf)
perform_mcnemar("K-Nearest Neighbors", preds_knn)
perform_mcnemar("Decision Tree", preds_tree)

--- McNemar Test: Red Neuronal vs Random Forest ---
P-value: 0.78210
No hay diferencia significativa.
Accuracy NN: 0.6571 | Accuracy Random Forest: 0.6519

--- McNemar Test: Red Neuronal vs K-Nearest Neighbors ---
P-value: 0.00280
Diferencia estadísticamente significativa.
Accuracy NN: 0.6571 | Accuracy K-Nearest Neighbors: 0.6097

--- McNemar Test: Red Neuronal vs Decision Tree ---
P-value: 0.00237
Diferencia estadísticamente significativa.
Accuracy NN: 0.6571 | Accuracy Decision Tree: 0.6004



---
## 6. Grid Search Manual + MLflow

Hasta ahora entrenamos un modelo con una sola configuración de hiperparámetros. Pero, ¿cómo sabemos que esa configuración es la **óptima**?

### ¿Qué es un Grid Search?
Un *grid search* es una búsqueda exhaustiva sobre un espacio de hiperparámetros definido. En lugar de adivinar, probamos **todas las combinaciones posibles** y nos quedamos con la que produce el mejor resultado en validación.

### ¿Qué es MLflow?
MLflow es una plataforma de rastreo de experimentos (*experiment tracking*). Cada vez que entrenamos un modelo con una configuración diferente, MLflow guarda automáticamente:
- Los **hiperparámetros** usados (lr, dropout, etc.)
- Las **métricas** obtenidas (val_accuracy, val_loss)
- Los **artefactos** generados (el modelo en sí)

Esto nos permite comparar experimentos de forma sistemática y reproducible, sin depender de notas o nombres de archivos.

In [8]:
# Instalación de MLflow
import subprocess
subprocess.run(['pip', 'install', 'mlflow', '--quiet'], check=True)
print('MLflow instalado correctamente.')

MLflow instalado correctamente.


In [9]:
import os
import mlflow
import mlflow.pytorch
import itertools
import time

# Permitir almacenamiento y conectar directamente al servidor UI abierto en el puerto 5001
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("chess_nn_grid_search")

print("MLflow conectado exitosamente al servidor en http://127.0.0.1:5001")

MLflow conectado exitosamente al servidor en http://127.0.0.1:5001


In [10]:
# ── Espacio de hiperparámetros ───────────────────────────────────────────────
param_grid = {
    'lr':           [1e-3, 5e-4],
    'hidden_size':  [64, 128],
    'dropout':      [0.3, 0.5],
    'weight_decay': [1e-4, 1e-3],
}

# Generar todas las combinaciones
keys   = list(param_grid.keys())
values = list(param_grid.values())
combinations = list(itertools.product(*values))
print(f'Total de combinaciones a evaluar: {len(combinations)}')

# ── Modelo genérico para el grid search ─────────────────────────────────────
class ChessNN_GS(nn.Module):
    def __init__(self, input_size, hidden_size, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.BatchNorm1d(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.6),
            nn.Linear(hidden_size // 2, 2)
        )
    def forward(self, x):
        return self.net(x)

EPOCHS_GS = 40  # Menos épocas para que el grid search sea rápido
results_gs = []

for idx, combo in enumerate(combinations):
    params = dict(zip(keys, combo))
    print(f'[{idx+1}/{len(combinations)}] Probando: {params}')

    with mlflow.start_run(run_name=f'run_{idx+1:02d}'):
        # Registrar hiperparámetros
        mlflow.log_params(params)
        mlflow.log_param('epochs', EPOCHS_GS)

        # Construir modelo y optimizador
        m   = ChessNN_GS(X.shape[1], params['hidden_size'], params['dropout'])
        opt = optim.Adam(m.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])

        # Entrenamiento
        m.train()
        for epoch in range(EPOCHS_GS):
            for bX, by in train_loader:
                opt.zero_grad()
                out  = m(bX)
                loss = criterion(out, by)
                loss.backward()
                opt.step()

        # Evaluación en validación
        m.eval()
        val_loss_total, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for bX, by in test_loader:
                out   = m(bX)
                loss  = criterion(out, by)
                val_loss_total += loss.item() * bX.size(0)
                _, preds = torch.max(out, 1)
                val_correct  += (preds == by).sum().item()
                val_total    += by.size(0)

        val_acc  = val_correct / val_total
        val_loss = val_loss_total / val_total

        # Registrar métricas en MLflow
        mlflow.log_metric('val_accuracy', val_acc)
        mlflow.log_metric('val_loss', val_loss)

        results_gs.append({**params, 'val_accuracy': val_acc, 'val_loss': val_loss})
        print(f'   val_acc={val_acc:.4f}  val_loss={val_loss:.4f}')

print('\n Grid search completo!')

Total de combinaciones a evaluar: 16
[1/16] Probando: {'lr': 0.001, 'hidden_size': 64, 'dropout': 0.3, 'weight_decay': 0.0001}
   val_acc=0.6529  val_loss=0.6182
🏃 View run run_01 at: http://127.0.0.1:5001/#/experiments/1/runs/6b03729ab7964ba2bb6d00c41074e1f3
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
[2/16] Probando: {'lr': 0.001, 'hidden_size': 64, 'dropout': 0.3, 'weight_decay': 0.001}
   val_acc=0.6447  val_loss=0.6152
🏃 View run run_02 at: http://127.0.0.1:5001/#/experiments/1/runs/80d98a3149244052a59f113e858d3ea1
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
[3/16] Probando: {'lr': 0.001, 'hidden_size': 64, 'dropout': 0.5, 'weight_decay': 0.0001}
   val_acc=0.6447  val_loss=0.6165
🏃 View run run_03 at: http://127.0.0.1:5001/#/experiments/1/runs/504af1cae0b34c49b9f806607cbb5ddd
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
[4/16] Probando: {'lr': 0.001, 'hidden_size': 64, 'dropout': 0.5, 'weight_decay': 0.001}
   val_acc=0.6540  val_los

In [11]:
# ── Resumen de resultados ─────────────────────────────────────────────────────
df_gs = pd.DataFrame(results_gs).sort_values('val_accuracy', ascending=False)
print('Top 5 configuraciones por val_accuracy:')
print(df_gs.head().to_string(index=False))

best = df_gs.iloc[0]
print(f'\nMejor configuración encontrada:')
print(f'  lr           = {best["lr"]}')
print(f'  hidden_size  = {int(best["hidden_size"])}')
print(f'  dropout      = {best["dropout"]}')
print(f'  weight_decay = {best["weight_decay"]}')
print(f'  val_accuracy = {best["val_accuracy"]:.4f}')
print()
print('TIP: Actualiza estos valores en red-neuronal/params.yaml y ejecuta `dvc repro`.')

Top 5 configuraciones por val_accuracy:
    lr  hidden_size  dropout  weight_decay  val_accuracy  val_loss
0.0005          128      0.3        0.0001      0.662204  0.616736
0.0005          128      0.5        0.0010      0.660144  0.616551
0.0010          128      0.5        0.0001      0.658084  0.616098
0.0005          128      0.3        0.0010      0.657055  0.618893
0.0010          128      0.3        0.0010      0.656025  0.617750

Mejor configuración encontrada:
  lr           = 0.0005
  hidden_size  = 128
  dropout      = 0.3
  weight_decay = 0.0001
  val_accuracy = 0.6622

TIP: Actualiza estos valores en red-neuronal/params.yaml y ejecuta `dvc repro`.


---
## 7. DVC — Pipeline Reproducible

### ¿Qué problema resuelve DVC?
Nuestro proceso de entrenamiento tiene dependencias encadenadas:
```
games.csv  →  preprocesamiento  →  datos procesados  →  entrenamiento  →  métricas
```
Si cambia `games.csv`, hay que volver a correr todo desde el principio. DVC (Data Version Control) **rastrea estas dependencias** y sólo re-ejecuta las etapas que realmente necesitan actualizarse.

Además, versiona los archivos de datos grandes (como `games.csv` o `data/processed.pkl`) con Git sin almacenarlos dentro del repositorio, guardándolos en un *remote* (en nuestro caso, una carpeta local).

### Estructura del pipeline (`dvc.yaml`)
```
┌─────────────────────────────────────────────┐
│  Etapa 1: preprocess                        │
│  deps: games.csv, chess_random_forest.joblib│
│  outs: data/processed.pkl                   │
└───────────────────┬─────────────────────────┘
                    │
┌───────────────────▼─────────────────────────┐
│  Etapa 2: train                             │
│  deps: data/processed.pkl                  │
│  params: params.yaml (lr, dropout, ...)    │
│  outs: models/best_model.pth               │
└───────────────────┬─────────────────────────┘
                    │
┌───────────────────▼─────────────────────────┐
│  Etapa 3: evaluate                          │
│  deps: models/best_model.pth               │
│  metrics: metrics/scores.json              │
└─────────────────────────────────────────────┘
```

In [12]:
# Instalación de DVC
import subprocess
subprocess.run(['pip', 'install', 'dvc', '--quiet'], check=True)
print('DVC instalado correctamente.')

# Verificar versión
result = subprocess.run(['dvc', '--version'], capture_output=True, text=True)
print(f'DVC version: {result.stdout.strip()}')

DVC instalado correctamente.
DVC version: 3.67.1


In [16]:
import subprocess
import os

# Nos posicionamos en red-neuronal/ para ejecutar los comandos DVC
NN_DIR = os.path.abspath('.')  # ya estamos en red-neuronal/ al ejecutar el notebook

def run_cmd(cmd, cwd=None, check=False):
    """Ejecuta un comando de shell y muestra el output."""
    result = subprocess.run(
        cmd, shell=True, capture_output=True, text=True,
        cwd=cwd or NN_DIR
    )
    output = result.stdout + result.stderr
    if output.strip():
        print(output.strip())
    return result

# ── 1. Inicializar DVC (si no existe aún) ────────────────────────────────────
print('=== dvc init ===')
run_cmd('dvc init --no-scm')
# Nota: --no-scm porque el .git ya está en el directorio raíz del proyecto,
# no dentro de red-neuronal/. Si quieres integrarlo con git del proyecto
# raíz, quita --no-scm y ejecuta desde allá.

print()

# ── 2. Configurar un remote LOCAL ────────────────────────────────────────────
# Esto es el 'almacén' donde DVC guardará los archivos versionados
dvc_remote_path = os.path.join(NN_DIR, '..', '..', 'dvc_remote_storage')
dvc_remote_path = os.path.abspath(dvc_remote_path)
os.makedirs(dvc_remote_path, exist_ok=True)

print(f'=== Configurando remote local en: {dvc_remote_path} ===')
run_cmd(f'dvc remote add -d myremote "{dvc_remote_path}"')
run_cmd('dvc remote list')

=== dvc init ===
ERROR: failed to initiate DVC - '.dvc' exists. Use `-f` to force.

=== Configurando remote local en: c:\Users\alvar\.gemini\antigravity\worktrees\PROYECTO_FIA\dvc_remote_storage ===
Setting 'myremote' as a default remote.
ERROR: configuration error - config file error: remote 'myremote' already exists. Use `-f|--force` to overwrite it.
myremote        
c:\Users\alvar\.gemini\antigravity\worktrees\PROYECTO_FIA\dvc_remote_storage   
(default)


CompletedProcess(args='dvc remote list', returncode=0, stdout='myremote        \nc:\\Users\\alvar\\.gemini\\antigravity\\worktrees\\PROYECTO_FIA\\dvc_remote_storage   \n(default)\n', stderr='')

In [24]:
# ── 3. Ejecutar el pipeline completo ─────────────────────────────────────────
print('=== dvc repro ===')
print('Ejecutando el pipeline de 3 etapas...')
print()
run_cmd('dvc repro')

print()
print('=== Métricas finales (dvc metrics show) ===')
run_cmd('dvc metrics show')

=== dvc repro ===
Ejecutando el pipeline de 3 etapas...

Stage 'preprocess' didn't change, skipping
Running stage 'train':
> python scripts/train_best.py
[train] Cargando datos preprocesados...
[train] Leyendo params.yaml...
[train] HiperparÃ¡metros: lr=0.01, dropout=0.4, hidden=128, wd=0.001, epochs=60
[train] Entrenando...
  Ã‰poca 10/60 â€” loss: 0.6169 acc: 0.6506
  Ã‰poca 20/60 â€” loss: 0.6181 acc: 0.6573
  Ã‰poca 30/60 â€” loss: 0.6146 acc: 0.6516
  Ã‰poca 40/60 â€” loss: 0.6138 acc: 0.6570
  Ã‰poca 50/60 â€” loss: 0.6213 acc: 0.6418
  Ã‰poca 60/60 â€” loss: 0.6165 acc: 0.6511
[train] Modelo guardado en: c:\Users\alvar\.gemini\antigravity\worktrees\PROYECTO_FIA\pytorch-training-analysis-report\red-neuronal\scripts\..\models\best_model.pth
Updating lock file 'dvc.lock'

Running stage 'evaluate':
> python scripts/evaluate.py
[evaluate] Cargando datos y modelo...
[evaluate] MÃ©tricas del modelo final:
  Accuracy : 0.6468
  F1-Score : 0.6430
[evaluate] Guardadas en: c:\Users\alvar\.

CompletedProcess(args='dvc metrics show', returncode=0, stdout='Path                 accuracy    f1_score\nmetrics\\scores.json  0.6468      0.643\n', stderr='')

In [22]:
# ── 4. Demostración de reproducibilidad ──────────────────────────────────────
print('=== dvc status (sin cambios) ===')
run_cmd('dvc status')
print()
print('dvc status mostrará "Data and pipelines are up to date" si no hubo cambios.')
print()
print('PRUEBA: Cambia un valor en params.yaml (por ejemplo, lr: 0.0005)')
print('y vuelve a ejecutar `dvc repro`. Solo las etapas "train" y "evaluate"')
print('se re-ejecutarán — "preprocess" quedará cacheada.')
print()

# Leer y mostrar métricas finales como tabla
import json
with open('metrics/scores.json', 'r') as fh:
    scores = json.load(fh)
print('Métricas finales del modelo:')
print(f"  Accuracy : {scores['accuracy']:.4f}")
print(f"  F1-Score : {scores['f1_score']:.4f}")

=== dvc status (sin cambios) ===
Data and pipelines are up to date.

dvc status mostrará "Data and pipelines are up to date" si no hubo cambios.

PRUEBA: Cambia un valor en params.yaml (por ejemplo, lr: 0.0005)
y vuelve a ejecutar `dvc repro`. Solo las etapas "train" y "evaluate"
se re-ejecutarán — "preprocess" quedará cacheada.

Métricas finales del modelo:
  Accuracy : 0.6601
  F1-Score : 0.6602
